# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

Each record set, field, and column is referenced by its `@id`.
Let's inspect the record sets and their fields.

In [ ]:
# List all record sets with their @id
record_sets = dataset.metadata.recordSets
print('Record Sets Found:')
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '')}")

# For each record set, list its fields by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']} - {rs.get('name', '')}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  Field @id: {f['@id']} | Name: {f.get('name', '')} | DataType: {f.get('dataType', '')}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use each record set's `@id` as reference.

Note: For this dataset, there are three main tabular record sets corresponding to tables described in the metadata:
- Table 1: Clinical features
- Table 2: Hormone levels/adrenal imaging
- Table 3: Genetic variants
Let's extract all and display their columns.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns of each DataFrame
for record_set_id in record_set_ids:
    print(f"\nData columns in RecordSet {record_set_id}:")
    print(dataframes[record_set_id].columns.tolist())
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

For demonstration, we use Table 2 (hormone/clinical measurement record set).

**Note:** Please update the variable names below as appropriate for your exploration using column `@id`s.

In [ ]:
# Identify Table 2 record set by @id:
table2_record_set_id = None
for rs in record_sets:
    if 'hormone' in rs.get('name', '').lower() or 'table 2' in rs.get('name', '').lower() or 'Hormone' in rs.get('description', ''):
        table2_record_set_id = rs['@id']
        break

# If not directly found, default to second record set
if not table2_record_set_id and len(record_set_ids) > 1:
    table2_record_set_id = record_set_ids[1]

df_table2 = dataframes[table2_record_set_id]

# Find a numeric column (e.g., hormone concentration -- demo below)
numeric_field_id = None
for col in df_table2.columns:
    if 'concentration' in col.lower() or 'level' in col.lower() or 'value' in col.lower():
        numeric_field_id = col
        break

# If not found, use the first numeric column
if not numeric_field_id:
    numeric_cols = df_table2.select_dtypes(include=np.number).columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]

print(f"Numeric field selected for analysis (@id): {numeric_field_id}")

# Filter records with value > threshold
threshold = df_table2[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = df_table2[df_table2[numeric_field_id] > threshold] if numeric_field_id else df_table2
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical column (e.g., imaging result or finding)
group_field = None
for col in df_table2.columns:
    if 'finding' in col.lower() or 'type' in col.lower() or 'result' in col.lower():
        group_field = col
        break

if group_field and numeric_field_id:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset (e.g., hormone level distribution, clinical feature counts, etc.).

In [ ]:
if numeric_field_id:
    plt.figure(figsize=(6,4))
    plt.hist(df_table2[numeric_field_id].dropna(), bins=10, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if group_field and numeric_field_id:
    plt.figure(figsize=(8,4))
    grouped_df = df_table2.groupby(group_field)[numeric_field_id].mean()
    grouped_df.plot(kind='bar', color='coral')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to explore the FAIR^2 dataset package on genotype–phenotype heterogeneity in NC-21OHD.

- We loaded dataset metadata and tabular records using their Croissant schema `@id`s for record sets and fields.
- We extracted and reviewed clinical, hormone, and genetic information via DataFrames.
- Applied basic filtering, normalization, and group-by analysis.
- Visualized distributions and relationships among measurements.

**This dataset is extremely valuable for case-based clinical and academic investigation, but its limited scope calls for careful interpretation. Please refer to column and record set `@id`s for robust downstream processing.**